### Middlewars
Middlewares are used to cutomize the input or output, or adding checks, rate limit, summarization etc. There are different types of middlewares which are used for different purpose.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="groq:openai/gpt-oss-20b",
        trigger=("messages", 5),
        keep=("messages", 4),
        )
    ]
)

In [11]:
### Run with Thread id
config={"configurable":{"thread_id":"test-1"}}

questions = [
    "what is 2+2?",
    "what is 3*5",
    "what is 55-33",
    "what is 98+12",
    "what is 78*2",
    "what is 11*11",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"response: {response}")
    print(f"Length of response: {len(response['messages'])}")

response: {'messages': [HumanMessage(content='what is 2+2?what is 3*5what is 55-33what is 98+12what is 78*2what is 11*11', additional_kwargs={}, response_metadata={}, id='3b6c81da-d8d1-438b-af3c-ce91e4ac28d2'), AIMessage(content='- 2\u202f+\u202f2\u202f=\u202f**4**  \n- 3\u202f×\u202f5\u202f=\u202f**15**  \n- 55\u202f−\u202f33\u202f=\u202f**22**  \n- 98\u202f+\u202f12\u202f=\u202f**110**  \n- 78\u202f×\u202f2\u202f=\u202f**156**  \n- 11\u202f×\u202f11\u202f=\u202f**121**', additional_kwargs={'reasoning_content': 'The user asks a series of arithmetic problems: "what is 2+2? what is 3*5 what is 55-33 what is 98+12 what is 78*2 what is 11*11". They want the answers. Probably just give the results. Ensure correct answers: 2+2=4, 3*5=15, 55-33=22, 98+12=110, 78*2=156, 11*11=121. Provide them clearly.\n\nWe need to respond in a helpful manner. Provide each answer. Maybe format as bullet list or separate lines. No extra commentary.'}, response_metadata={'token_usage': {'completion_tokens': 22

### SummarizationMiddleware with Messages

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-20b")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 5),
            keep=("messages", 4),
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

questions = [
    "what is 2+2?",
    "what is 3*5?",
    "what is 55-33?",
    "what is 98+12?",
    "what is 78*2?",
    "what is 11*11?"
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

    print(f"Question: {q}")
    print(f"Answer: {response['messages'][-1].content}")
    print(f"Length of response: {len(response['messages'])}")
    print("-------------------------")

Question: what is 2+2?
Answer: 2 + 2 = 4
Length of response: 2
-------------------------
Question: what is 3*5?
Answer: 3 × 5 = 15
Length of response: 4
-------------------------
Question: what is 55-33?
Answer: 55 – 33 = 22
Length of response: 6
-------------------------
Question: what is 98+12?
Answer: 98 + 12 = 110
Length of response: 6
-------------------------
Question: what is 78*2?
Answer: 78 × 2 = 156
Length of response: 6
-------------------------
Question: what is 11*11?
Answer: 11 × 11 = 121
Length of response: 6
-------------------------


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens", 1000),
            keep=("tokens", 400)
        )
    ]
)
config = {"configurable": {"thread_id": "test-2"}}
questions = [
    "what is 2+2?",
    "what is 7*5?",
    "what is 5-2?",
    "what is 2*8?",
    "what is 10*10?",
    "what is 10-5?"
]

for q in questions:
    response = agent.invoke(
            {"messages": [HumanMessage(content=q)]},
            config=config
        )
    print(f"Response: {response}")
    print(f"length: {len(response['messages'])}")

Response: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='79a9c68f-e978-4eec-a58d-eeb70534ee4e'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks: "what is 2+2?" Simple math. Provide answer: 4. Probably no extra explanation needed. But can also give explanation.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 78, 'total_tokens': 130, 'completion_time': 0.060743773, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.005161133, 'prompt_tokens_details': None, 'queue_time': 0.378831296, 'total_time': 0.065904906}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17f41-65b8-7793-8537-85b94d49ad32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 52, 'to

In [7]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

model = "groq:openai/gpt-oss-20b"

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 1000),
            keep=("tokens", 400),
        )
    ],
)

config = {"configurable": {"thread_id": "test-2"}}

questions = [
    "what is 2+2?",
    "what is 7*5?",
    "what is 5-2?",
    "what is 2*8?",
    "what is 10*10?",
    "what is 10-5?",
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config,
    )

    print(f"Question: {q}")
    print(f"Answer: {response['messages'][-1].content}")
    print(f"Message count: {len(response['messages'])}")
    print("--------------------")


Question: what is 2+2?
Answer: 2 + 2 = 4.
Message count: 2
--------------------
Question: what is 7*5?
Answer: 7 × 5 = 35.
Message count: 4
--------------------
Question: what is 5-2?
Answer: 5 − 2 = 3.
Message count: 6
--------------------
Question: what is 2*8?
Answer: 2 × 8 = 16.
Message count: 8
--------------------
Question: what is 10*10?
Answer: 10 × 10 = 100.
Message count: 10
--------------------
Question: what is 10-5?
Answer: 10 − 5 = 5.
Message count: 12
--------------------


### HumanInTheLoopMiddleware

In [44]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

config = {"configurable": {"thread_id": "place_order_test_4"}}
@tool
def Search_Product(product_name: str, user_budget: int)-> str:
    """Use this fuction to search the product under the max price """
    return f"{product_name} is available and its price is in your budget, under the {user_budget}."

@tool
def Place_Order(name: str, price: str, deliveryTime: str)->str:
    """use this tool to place the order"""
    return f"Your Order is placed {name} with this price: {price}. Note: it will be deliver in {deliveryTime}. Thankyou for shopping."


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    tools=[Search_Product, Place_Order],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on= {
                "Place_Order": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "Search_Product": False
            }
        )
    ]
)

Messages = [SystemMessage(content="""
You are an expert online shopping agent.
Always use the actual tool result when confirming an order.
Never claim an order was placed with details that differ from the tool result.
"""),
              HumanMessage(content="Find me a wireless mouse under $30.")]

response = agent.invoke({"messages": Messages}, config)
print(response["messages"][-1].content)
print(response.get("__interrupt__"))
print("--------------------------------")

response2 = agent.invoke(
    {"messages": [HumanMessage(content="My name is Asjiad Farooq, I want to order the Logitech M185 Wireless Mouse for $19.99. Deliver it next Friday.")]},
    config
)
print(response2["messages"][-1].content)
print(response2.get("__interrupt__"))

if (response2.get("__interrupt__")):
    # result = agent.invoke(
    #     Command(
    #         resume={
    #             "decisions": [
    #                 {
    #                     "type": "approve"
    #                 }
    #             ]
    #         }
    #     ),
    #     config
    # )

    # result = agent.invoke(
    #     Command(
    #         resume={
    #             "decisions": [
    #                 {
    #                     "type": "reject",
    #                     "message": "Order is declined successfully"
    #                 }
    #             ]
    #         }
    #     ),
    #     config
    # )

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "Place_Order",
                            "args": {
                                "name": "Dell Mouse",
                                "price": "15",
                                "deliveryTime": "7 days"
                            }
                        }
                    }
                ]
            }
        ),
        config
    )


    print(result["messages"])

Sure! I’ve checked the inventory and found a wireless mouse that fits your $30 budget.  

**Product:** Wireless Mouse  
**Price:** Under $30  
**Availability:** In stock

If you’d like more details (brand, model, features, or a few specific options), just let me know and I can narrow it down for you!
None
--------------------------------

[Interrupt(value={'action_requests': [{'name': 'Place_Order', 'args': {'deliveryTime': 'next Friday', 'name': 'Asjiad Farooq', 'price': '$19.99'}, 'description': "Tool execution requires approval\n\nTool: Place_Order\nArgs: {'deliveryTime': 'next Friday', 'name': 'Asjiad Farooq', 'price': '$19.99'}"}], 'review_configs': [{'action_name': 'Place_Order', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='5383629daa184b3447599fa2a1bd2a3a', response_schema=None)]
[SystemMessage(content='\nYou are an expert online shopping agent.\nAlways use the actual tool result when confirming an order.\nNever claim an order was placed with details that differ fr

### Model Call Limit Middleware

In [45]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.messages import HumanMessage


config = {"configurable": {"thread_id": "model_limit_1"}}
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware= [
        ModelCallLimitMiddleware(
            thread_limit=3,
            run_limit=2,
            exit_behavior="end"
        )
    ]
)
Messages = [
    "what is 2+2?",
    "what is 5+4?",
    "what is 5-3?",
    "what is 8*6",
    "what is 4*20?",
    "what is 8*8?"
]

for msg in Messages:
    response = agent.invoke({"messages": [HumanMessage(content=msg)]}, config)
    print(response["messages"][-1].content)

2 + 2 = 4.
5 + 4 = 9.
5 − 3 = 2.
Model call limits exceeded: thread limit (3/3)
Model call limits exceeded: thread limit (3/3)
Model call limits exceeded: thread limit (3/3)


### Tool call Limit Middleware

In [55]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.messages import HumanMessage, SystemMessage
from langchain.messages import HumanMessage
from langchain.tools import tool


config = {"configurable": {"thread_id": "model_limit_1"}}

@tool
def Get_result(resultpattern) -> str:
    """Get result format of any calculation question from this tool"""
    return f"Your Calculation result is: {resultpattern}"
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    tools=[Get_result],
    middleware= [
        ToolCallLimitMiddleware(
            thread_limit=3,
            run_limit=2,
            exit_behavior="end"
        )
    ]
)
Messages = [
    "what is 2+2?",
    "what is 5+4?",
    "what is 5-3?",
    "what is 8*6",
    "what is 4*20?",
    "what is 8*8?"
    ]
for msg in Messages:
    response = agent.invoke(
    {"messages": [
        SystemMessage(content="You must use the Get_result tool for every calculation. Do not answer without using it."),
        HumanMessage(content=msg)
    ]},
        config
    )
    print(response["messages"][-1].content)

Your Calculation result is: 2+2
Your Calculation result is: 5+4
Your Calculation result is: 5-3
Tool call limit reached: thread limit exceeded (4/3 calls).
I’m sorry, but I’m unable to perform the calculation right now because I’ve reached the tool‑usage limit for this conversation.
I’m sorry, but I can’t complete that calculation right now.
